In [1]:
%load_ext autoreload
%autoreload 2

In [54]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy, plot_confusion_matrices, plot_distributions, plot_accuracy, plot_metric_heatmap, plot_precision_recall_curves, plot_cooccurrence, plot_error_distribution, GT_COLS, PRED_COLS, LABELS
from utils.tools import get_config, collect_jobs, prettify_table

In [3]:
def create_row_annotation(config):
    annotations = []

    if bool(config.get("use_instructions", 0)):
        annotations.append("with instructions")
    else:
        annotations.append("without instructions")

    n_demos = config.get("number_of_demonstrations")
    annotations.append(f"number of demonstrations: {n_demos}")
    if n_demos > 0:
        t = ""
        demo_type = config.get("type_of_demonstrations")
        match demo_type:
            case -1:
                t = "negative"
            case 0:
                t = "mixed"
            case 1:
                t = "positive"

        annotations.append(f"type of demonstrations: {t}")


    return "\n".join(annotations)

In [4]:
def conf_matrices(jobs):
    num_jobs = len(jobs)
    
    print("Number of completed jobs: " + str(num_jobs))
    
    fig, axes = plt.subplots(num_jobs, 3, figsize=(16, 5 * num_jobs))
    if num_jobs == 1:
        axes = [axes]
    
    fig.suptitle(jobs[0]["model"], fontsize=16)
    
    for i, job_info in enumerate(jobs):
        config = get_config(job_info["config_json"])
            
        df = pd.read_csv(job_info["result_csv"], sep=";")
        plot_confusion_matrices(df, axes[i], use_title=True if i == 0 else False, xlabel="Predicted" if i == (num_jobs - 1) else "", ylabel="True")
    
        axes[i][0].annotate(
            create_row_annotation(config),
            xy=(-0.2, 0.5),
            xycoords='axes fraction',
            fontsize=12,
            ha='right',
            va='center',
            rotation=90
        )
    return fig, axes

#fig, axes = conf_matrices(finished_jobs)
#fig.tight_layout(rect=[0, 0, 1, 0.98])
#plt.savefig("conf.png", pad_inches=0.3)
#plt.show()